# B-cell Epitope — phase 1 on Kaggle

Phase 1 runs internal 5-fold cross-validation on the B-cell data only. **SARS is
never opened**, which is exactly what keeps it a valid external test for phase 2.

Every measurement is recorded: per-fold train and validation metrics, pooled
out-of-fold predictions, and the full table of all eight hyperparameter candidates
per run.

## Nothing needs changing

Phase 1 is a single session. SVM was dropped from the sweep, and the four
remaining classifiers -- random forest, logistic regression, extra trees,
XGBoost -- handle DPC's 400 columns comfortably. It was SVM, whose cost grows
as n^2.3 and which the calibration wrapper fits four times per model fit,
that put phase 1 past Kaggle's 12 h cap.

The dataset ships inside the repository, so nothing needs attaching.

## Every session must run in the same environment

scikit-learn decides the fold split and every estimator; pandas decides the row
order feeding into it. Different library versions can produce **different folds**,
which shifts every number in the table at once. This is not hypothetical: running
the same commit, data and seed on a local machine with scikit-learn 1.9.1 and
pandas 3.0.6 gave random-forest ROC-AUC differing by up to **0.039** and PR-AUC SD
by **2.4×**, with nothing about the method changed.

The canonical environment is the Kaggle image phase 1 ran on:

| | |
|---|---|
| Python | 3.13.18 |
| numpy | 2.1.3 |
| pandas | 2.3.3 |
| scikit-learn | 1.6.1 |
| xgboost | 3.4.1 |

Cell 1 refuses to run on a clone that is not at `origin/master`; cell 2 compares
the live versions against that table and warns if Kaggle has refreshed its image.
The versions also go into every row of `experiment_summary.csv`, and the selection
step refuses to merge runs made under different ones.

## What the table shows you

31 rows per model, 124 in total, in six parts:

| Part | Rows | What it answers |
|---|---|---|
| 1 | 15 | which peptide representation works -- power set over **AAC**, **DPC**, **PROP**, **FLANK**, with **LEN** held on. **The only arm the one-standard-error rule selects from.** |
| 2 | 4 | what each step away from the peptide buys, retracing the literature's path from PROP+LEN. Its first two rungs reproduce **FS-2** and **FS-1** column for column. |
| 3 | 4 | whether protein-level information helps a good peptide model, on the fixed 67-column backbone AAC+PROP+FLANK+LEN |
| 4 | 4 | the raw signal in each protein-level group alone, PROTID in particular |
| 5 | 1 | what LEN contributes, by running the backbone without it |
| 6 | 3 | FS-1, FS-2, FS-3 exactly as published |

Groups are split by **what they are computed from**. Parts 1's four groups
come from the peptide or the residues flanking it, so each is usable on a
protein never seen in training. The groups in parts 2 to 4 come from the
whole protein or from the row's place in the dataset, so every peptide of a
protein shares the value: measured intra-class correlation with protein
identity is 1.000 for PROTGLOBAL, PROTLEN and PROTID, and 0.797 for POS.
Those are reported and never selected, so the frozen configuration cannot
contain one.

Part 6 doubles as a check on the whole rerun. Nothing in the redesign touched
the fixed paper profiles, so those 12 numbers should come back identical to
the earlier sweep. If they do not, something changed that was not meant to.

## Saving your results

`/kaggle/working` is wiped when a session ends. Before closing a session, use
**Save Version** and download the files the collection cell links — in particular
`experiment_summary.csv` and `internal_table.tsv`.

## What this notebook deliberately does not do

It does not choose a feature configuration. That step -- the one-standard-error
rule over the 60 part-1 runs (15 combinations x 4 classifiers) -- is run
afterwards on the summary file:

    python -u -m ML.select_configuration \
        phase1/experiment_summary.csv \
        --output frozen_configuration.json

Only part 1 enters that rule. Parts 2 to 5 carry `feature_set="kontras"` and are
filtered out before the groups are read, so no protein-level group can end up in
the frozen configuration.

Phase 2 is not here either. It gets built once that rule has chosen a
combination.

In [ ]:
# ---- 1. Get the pipeline ----------------------------------------------------
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/azhar416/B-Cell-Epitopes.git"
PROJECT_DIR = Path("/kaggle/working/B-Cell-Epitopes")

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    if PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"{PROJECT_DIR} exists but holds no ML pipeline. Remove it, or point "
            "PROJECT_DIR at the repository root."
        )
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    raise FileNotFoundError(f"Clone finished but {PROJECT_DIR}/ML/train.py is absent.")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


def _git(*arguments):
    return subprocess.run(
        ["git", "-C", str(PROJECT_DIR), *arguments],
        capture_output=True, text=True,
    ).stdout.strip()


# A clone left behind by an earlier step would be reused silently and run old
# code, so confirm it is the current master rather than assuming it.
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "fetch", "-q", "--depth", "1", "origin", "master"],
    capture_output=True, text=True,
)
head = _git("rev-parse", "HEAD")
remote = _git("rev-parse", "FETCH_HEAD") or _git("rev-parse", "origin/master")

print(f"Pipeline at {PROJECT_DIR}")
print(f"  commit in use : {head[:12]}")
print(f"  origin/master : {remote[:12] or 'could not be read'}")
if remote and head != remote:
    raise RuntimeError(
        "This clone is not at origin/master, so it would run outdated code. "
        f"In use {head[:12]}, remote {remote[:12]}. Delete {PROJECT_DIR} and "
        "re-run this cell."
    )
print("  -> up to date")
COMMIT = head


In [ ]:
# ---- 2. Dependencies and imports -------------------------------------------
import importlib.util

required = ("numpy", "pandas", "sklearn", "imblearn", "joblib", "matplotlib", "xgboost")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print(f"Installing: {missing}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_DIR / "ML" / "requirements.txt")]
    )

import pandas as pd
from IPython.display import display, FileLink

from ML.CFG import CFG
from ML.collect_artifacts import collect_all
from ML.dataset import load_labeled_dataset
from ML.experiment_matrix import (
    ALL_COMBINATIONS,
    CONTRAST_COMBINATIONS,
    MODEL_TYPES,
    PAPER_RECONSTRUCTIONS,
    REPRODUCTION_CHECK_COMBINATIONS,
    build_experiment_matrix,
    report_environment,
    run_experiment_matrix,
)
from ML.summary_table import (
    build_internal_table,
    internal_header,
    read_summaries,
    render_internal_pretty,
    render_rows_tsv,
)

# scikit-learn decides the fold split and every estimator; pandas decides the
# row order feeding into it, so a version change can hand out different folds
# and shift every number in the table at once. This compares the live versions
# against the environment phase 1 was actually run in. They are also written
# into every row of experiment_summary.csv.
ENVIRONMENT_OK = report_environment()
if not ENVIRONMENT_OK:
    print()
    print("Kaggle has refreshed its image since phase 1 ran. Results produced")
    print("here cannot be merged with the earlier sessions; the selection step")
    print("will refuse to combine them. Either rerun every phase-1 session in")
    print("this new environment, or pin the versions with:")
    print(f"    !pip install -q -r {PROJECT_DIR}/ML/requirements.txt")


## Settings

In [ ]:
# ---- 3. Nothing here needs changing ----------------------------------------
from collections import Counter

# SVM is no longer swept, so phase 1 is a single session. The four remaining
# models handle DPC's 400 columns comfortably; it was SVM that did not.
SWEEP_MODELS = MODEL_TYPES

# Per-peptide prediction tables are the largest output (~1 M rows). Every
# pooled metric is computed from them, so they are collected by default.
COLLECT_PREDICTIONS = True

WORKING = Path("/kaggle/working")
RUN_DIR = WORKING / "ML-artifacts" / "phase1"
COLLECTED_DIR = WORKING / "collected" / "phase1"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = RUN_DIR / "experiment_summary.csv"

# The repository ships the datasets, so prefer them; an attached Kaggle
# dataset is only a fallback.
def _find_data_dir(filename):
    bundled = PROJECT_DIR / "dataset"
    if (bundled / filename).is_file():
        return bundled
    found = sorted(Path("/kaggle/input").rglob(filename))
    if len(found) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}; found {len(found)}. "
            "Attach the dataset, or set DATA_DIR by hand."
        )
    return found[0].parent

DATA_DIR = _find_data_dir("input_bcell.csv")

cfg = CFG(
    DATASET_NAME="bcell",
    TRAINING_MODE="same_dataset",     # phase 1: B-cell only, SARS untouched
    DATA_DIR=DATA_DIR,
    OUTPUT_DIR=RUN_DIR,
)
cfg.validate()

planned = build_experiment_matrix(SWEEP_MODELS)
arms = Counter(spec.feature_set for spec in planned)
papers = sum(c for name, c in arms.items() if name.startswith('perbandingan'))
print(f"models: {list(SWEEP_MODELS)}")
print(f"{len(planned)} runs = {len(planned) // len(SWEEP_MODELS)} configurations x "
      f"{len(SWEEP_MODELS)} models")
print(f"  part 1      {arms['proposed'] // len(SWEEP_MODELS):3d}  power set over AAC/DPC/PROP/FLANK, LEN always on")
print(f"                   the only arm the one-standard-error rule selects from")
print(f"  parts 2-5   {arms['kontras'] // len(SWEEP_MODELS):3d}  planned contrasts, reported but never selected")
print(f"  part 6      {papers // len(SWEEP_MODELS):3d}  FS-1/FS-2/FS-3 exactly as published")
print(f"mode={cfg.TRAINING_MODE} (SARS is not read)")
print(f"tuning={cfg.HYPERPARAMETER_TUNING} metric={cfg.HYPERPARAMETER_TUNING_METRIC} "
      f"candidates/model={cfg.TUNING_CANDIDATES_PER_MODEL}")
print(f"class_weight={cfg.CLASS_WEIGHT} scale_pos_weight={cfg.XGBOOST_SCALE_POS_WEIGHT} "
      f"smote={cfg.USE_SMOTE} seed={cfg.RANDOM_STATE}")
print(f"flank window={cfg.FLANK_WINDOW} split_sides={cfg.FLANK_SPLIT_SIDES}")
print(f"data={DATA_DIR}")
print(f"out={RUN_DIR}")

## Restarting a run that ran out of time

**Skip this cell unless that happened.** Phase 1 is one session of about 124
runs, well inside Kaggle's 12 h cap, so this is a safety net rather than part
of the normal path. Attach the timed-out session's saved output, set both
variables, and the runs that already finished are skipped instead of repeated.

`RESUME` decides whether to skip; `PREVIOUS_OUTPUT` is what makes the earlier
reports visible in the first place, since skipping works by finding a run's
report inside `RUN_DIR`. Setting one without the other does nothing useful, so
the cell says so rather than letting you lose the hours quietly.

In [ ]:
# ---- 4. Restart an interrupted session (normally skip) ---------------------
import shutil

# Example, when continuing a session that timed out:
#   PREVIOUS_OUTPUT = "/kaggle/input/bcell-phase1/ML-artifacts/phase1"
#   RESUME = True
PREVIOUS_OUTPUT = None
RESUME = False

if PREVIOUS_OUTPUT is not None:
    source = Path(PREVIOUS_OUTPUT)
    if not source.is_dir():
        raise FileNotFoundError(f"PREVIOUS_OUTPUT is not a directory: {source}")
    copied = 0
    for item in source.rglob("*"):
        if not item.is_file():
            continue
        target = RUN_DIR / item.relative_to(source)
        if target.exists():
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, target)
        copied += 1
    print(f"Restored {copied} file(s) into {RUN_DIR}")
    if not RESUME:
        print("WARNING: RESUME is False, so the restored runs will be repeated.")
elif RESUME:
    if not list(RUN_DIR.rglob("*_metrics.json")):
        print("WARNING: RESUME is True but RUN_DIR holds no reports, so nothing "
              "will be skipped. Set PREVIOUS_OUTPUT to the attached run directory.")
else:
    print("Fresh run; nothing to restore.")


## Check the data before spending hours on it

The cleaner drops overlength peptides, contradictory-label peptides, and exact
duplicates. These counts must match every other run of this pipeline; if they do
not, the input file differs and nothing below is comparable to earlier results.

Expected: **14,387 → 14,358 rows, 758 proteins, `{0: 10474, 1: 3884}`**.


In [ ]:
# ---- 5. Data sanity check --------------------------------------------------
bcell = load_labeled_dataset(
    "bcell", cfg.DATA_DIR,
    max_peptide_length=cfg.MAX_PEPTIDE_LENGTH, feature_set="proposed",
)
counts = bcell.target.value_counts().sort_index().to_dict()
proteins = int(bcell.parent_protein_id.nunique())
print(f"B-cell: {len(bcell):,} rows, {proteins:,} proteins, classes {counts}")

expected = (14358, 758, {0: 10474, 1: 3884})
actual = (len(bcell), proteins, counts)
if actual != expected:
    print(f"WARNING: expected {expected} but got {actual}. The input file differs "
          "from the one earlier results came from.")
else:
    print("Matches the reference cleaning counts.")

print("SARS is not read in phase 1, so it stays an untouched external test set.")


## Run

Driven by `ML.experiment_matrix`, the same module the command line uses, so Kaggle
and local runs execute identical code. The summary CSV is rewritten after every
run, so progress survives a crash.

Each run selects **one** hyperparameter set: all eight candidates are scored on
pooled out-of-fold predictions over the whole development set, and the winner is
held fixed across the five folds. The reported mean ± SD therefore describes a
single model instead of averaging five differently-tuned ones.


In [ ]:
# ---- 6. Run ----------------------------------------------------------------
import time

started = time.time()
matrix_rows = run_experiment_matrix(
    RUN_DIR,
    base_config=cfg,
    model_types=SWEEP_MODELS,
    resume=RESUME,
)
elapsed = time.time() - started

matrix_summary = pd.DataFrame(matrix_rows)
ok = int((matrix_summary["status"] == "success").sum())
print()
print(f"{len(matrix_summary)} runs in {elapsed / 3600:.2f} h -- "
      f"{ok} successful, {len(matrix_summary) - ok} failed")

failures = matrix_summary[matrix_summary["status"] != "success"]
if not failures.empty:
    display(failures[["experiment_id", "error"]])
display(FileLink(str(SUMMARY_PATH)))


## Did this session run in the same environment as the others?

scikit-learn assigns the folds and pandas orders the rows feeding it, so a
version bump hands out different folds and shifts every metric at once. Cell 2
checks the versions against the declared environment, but a summary written
before version recording existed cannot be checked that way.

This cell checks by result instead: point `PREVIOUS_SUMMARY` at another
session's `experiment_summary.csv` and the two cheapest power-set
combinations, which both sessions run, must come out the same. Needed
whenever a sweep is split across sessions, for example `nodpc` followed by
`dpconly`.

In [ ]:
# ---- 6b. Cross-session environment check -----------------------------------
PREVIOUS_SUMMARY = None   # Path("/kaggle/input/<dataset>/experiment_summary.csv")

# Random forest and extra trees sum over threads, so repeating the same fit
# differs at ~1e-8. A reassigned fold shifts a metric by ~1e-2. The gap
# between those two scales is what makes this check decisive.
NOISE_TOLERANCE = 1e-6
FOLD_CHANGE_SCALE = 1e-3

if PREVIOUS_SUMMARY is None:
    print("PREVIOUS_SUMMARY is not set. Fine for a single full sweep; set it")
    print("when combining sessions, or the merge is UNVERIFIED.")
else:
    previous = pd.read_csv(PREVIOUS_SUMMARY).set_index("experiment_id")
    current = pd.read_csv(SUMMARY_PATH).set_index("experiment_id")
    metric_columns = [
        column
        for column in current.columns
        if column.startswith(("mean_validation_", "std_validation_"))
        and column in previous.columns
    ]
    checked = [
        "__".join(("proposed", model, "__".join(combination)))
        for model in SWEEP_MODELS
        for combination in REPRODUCTION_CHECK_COMBINATIONS
    ]
    shared = [
        name for name in checked
        if name in previous.index and name in current.index
    ]
    if not shared:
        raise RuntimeError(
            "No run appears in both summaries, so nothing can be compared. "
            f"Expected any of: {checked}"
        )
    worst = (
        current.loc[shared, metric_columns].astype(float)
        - previous.loc[shared, metric_columns].astype(float)
    ).abs().max(axis=1).sort_values(ascending=False)
    print(f"Compared {len(shared)} runs over {len(metric_columns)} metric columns.")
    display(worst.rename("max |difference|").to_frame())
    largest = float(worst.max())
    print()
    print(f"Largest difference anywhere: {largest:.3e}")
    if largest <= NOISE_TOLERANCE:
        print("SAME ENVIRONMENT. The two summaries may be merged.")
    elif largest >= FOLD_CHANGE_SCALE:
        print("DIFFERENT FOLDS. Do NOT merge: rerun the whole sweep in one")
        print("environment, or the table mixes two fold assignments.")
    else:
        print("INCONCLUSIVE: above thread noise, below a fold change. Find")
        print("which columns moved before merging anything.")

## Collect every measurement

`ML.collect_artifacts` reads the summary as its index of runs and flattens the
hundreds of per-run files into long-format tables:

| File | One row per | Holds |
|---|---|---|
| `runs.csv` | run | pooled OOF, mean ± SD, train mean ± SD, protocol text |
| `fold_metrics.csv` | run × fold | train and validation metrics, fold sizes |
| `hyperparameter_candidates.csv` | run × candidate | every candidate's score, so the winner's margin is visible |
| `oof_predictions.csv.gz` | run × peptide | the probabilities every pooled metric is computed from |

The candidate table is the one worth keeping: `selected_hyperparameters` on its own
hides how close the runner-up was.

Run directories are rebuilt relative to the summary file rather than from the
absolute paths recorded during the run, so these tables still collect correctly
after the output is downloaded and moved.


In [ ]:
# ---- 7. Collect metrics ----------------------------------------------------
manifest = collect_all(
    [SUMMARY_PATH], COLLECTED_DIR, include_predictions=COLLECT_PREDICTIONS
)

print(f"Collected {manifest['runs_collected']} run(s) into {COLLECTED_DIR}\n")
for name, info in manifest["tables"].items():
    size_mb = (COLLECTED_DIR / info["file"]).stat().st_size / 1048576
    print(f"  {name:<28} {info['rows']:>9,} rows  {size_mb:>7.1f} MB  {info['file']}")
    display(FileLink(str(COLLECTED_DIR / info["file"])))

if manifest["runs_missing_report"]:
    print(f"\nNo readable report for {len(manifest['runs_missing_report'])} run(s):")
    for experiment_id in manifest["runs_missing_report"]:
        print(f"  {experiment_id}")

display(FileLink(str(COLLECTED_DIR / "manifest.json")))


## The internal comparison table

One row per run: the 15 feature combinations, then the three comparison-paper
feature sets, for each model this session ran. Metrics are `mean ± SD` across the
five folds.

Only the feature engineering is taken from each comparison study — the classifier,
folds, seed, tuning budget and imbalance handling are ours throughout. Labelling
those rows with the original method names would misreport those studies, so they
are named as feature sets (`FS-1`, `FS-2`, `FS-3`).

The printed grid is for reading here. The `.tsv` file is the one that pastes
cleanly into a spreadsheet, tab-separated.


In [ ]:
# ---- 8. Internal comparison table -----------------------------------------
# model_order fixes the block order and leaves out any classifier this sweep
# did not run, rather than printing it as a blank block.
internal_rows = build_internal_table(
    read_summaries([SUMMARY_PATH]), model_order=SWEEP_MODELS
)
print(render_internal_pretty(internal_rows))

TSV_PATH = COLLECTED_DIR / "internal_table.tsv"
TSV_PATH.write_text(render_rows_tsv(internal_header(), internal_rows), encoding="utf-8")
filled = sum(1 for row in internal_rows if row[2])
print(f"\n{filled}/{len(internal_rows)} rows have internal metrics.")
print("Download this and paste it into the spreadsheet:")
display(FileLink(str(TSV_PATH)))


## Drop the model bundles

Every run saves fitted model bundles. Across a session the random forest and extra
trees bundles reach several gigabytes, which can exceed Kaggle's output limit and
makes downloading impractical.

The bundles are not measurements and everything above has already been collected,
so this removes them. Run it only after the collection cell succeeded.


In [ ]:
# ---- 9. Reclaim space ------------------------------------------------------
DROP_MODEL_BUNDLES = True
DROP_PLOTS = False          # per-run PNGs; keep them if you want the curves

def _purge(pattern):
    files = list(RUN_DIR.rglob(pattern))
    freed = sum(path.stat().st_size for path in files)
    for path in files:
        path.unlink()
    return len(files), freed / 1048576

def _tree_size(root):
    return sum(p.stat().st_size for p in root.rglob("*") if p.is_file()) / 1048576

print(f"Before: {_tree_size(RUN_DIR):,.1f} MB in {RUN_DIR}")
if DROP_MODEL_BUNDLES:
    count, freed = _purge("*.joblib")
    print(f"  removed {count} model bundles ({freed:,.1f} MB)")
if DROP_PLOTS:
    count, freed = _purge("*.png")
    print(f"  removed {count} plots ({freed:,.1f} MB)")
print(f"After:  {_tree_size(RUN_DIR):,.1f} MB")
print(f"Collected tables: {_tree_size(COLLECTED_DIR):,.1f} MB in {COLLECTED_DIR}")


## Before closing this session

1. **Save Version**, so Kaggle keeps the output.
2. Download at least these two, from the links above:
   - `experiment_summary.csv` -- needed to choose a feature configuration
   - `internal_table.tsv` -- the table to paste into the spreadsheet

Then choose the configuration outside this notebook:

    python -u -m ML.select_configuration \
        phase1/experiment_summary.csv \
        --output frozen_configuration.json

That is the one-standard-error rule over the 60 part-1 runs. It is kept out of
here on purpose: it needs every classifier finished, and it is cheap to rerun
while the sweep is not.

## What these numbers do and do not support

The internal metrics are **not** an independent performance estimate. Candidates
are selected from the same out-of-fold predictions the metrics are computed from,
so they are model-selection results. Every report says so in
`hyperparameter_tuning_warning`; keep that wording in the paper.

What phase 1 *is* good for: comparing feature combinations and classifiers under
one identical protocol — same folds, same seed, same tuning budget of eight
candidates per model, same imbalance handling.

SARS is untouched here, so it stays a genuine external test for phase 2.
